# TCGA-BRCA Clinical XML Endpoint-Target Prep V1 Review

This notebook reviews the saved clinical XML endpoint-target prep v1 outputs from disk only. It does not parse raw XML files, freeze the endpoint, add treatment detail, or perform modeling.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'endpoint-prep'
    / 'tcga_brca_endpoint_target_prep_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest endpoint-target prep pointer not found: {latest_pointer_path}. Run the endpoint-prep script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
patient_fields_path = repo_root / latest_pointer['clinical_xml_patient_endpoint_fields_tsv']
coverage_path = repo_root / latest_pointer['clinical_xml_followup_version_coverage_tsv']
endpoint_prep_path = repo_root / latest_pointer['endpoint_target_prep_v1_tsv']
overlap_audit_path = repo_root / latest_pointer['endpoint_target_prep_v1_overlap_audit_tsv']
summary_path = repo_root / latest_pointer['endpoint_target_prep_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

patient_fields_df = read_tsv(patient_fields_path)
coverage_df = read_tsv(coverage_path)
endpoint_prep_df = read_tsv(endpoint_prep_path)
overlap_audit_df = read_tsv(overlap_audit_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if run_log.get('status') != 'completed':
    raise ValueError('run_log.json does not report status == completed.')
if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if patient_fields_df.empty:
    raise ValueError('clinical_xml_patient_endpoint_fields.tsv contains no rows.')
if endpoint_prep_df.empty:
    raise ValueError('endpoint_target_prep_v1.tsv contains no rows.')
if overlap_audit_df.shape[0] != endpoint_prep_df.shape[0]:
    raise ValueError('Overlap audit row count did not match endpoint-target prep row count.')


In [2]:
patient_fields_review_path = results_root / '97_clinical_xml_patient_endpoint_fields.tsv'
coverage_review_path = results_root / '98_clinical_xml_followup_version_coverage.tsv'
endpoint_prep_review_path = results_root / '99_endpoint_target_prep_v1.tsv'
overlap_audit_review_path = results_root / '100_endpoint_target_prep_v1_overlap_audit.tsv'
summary_review_path = results_root / '101_endpoint_target_prep_v1_summary.tsv'

patient_fields_df.to_csv(patient_fields_review_path, sep='\t', index=False)
coverage_df.to_csv(coverage_review_path, sep='\t', index=False)
endpoint_prep_df.to_csv(endpoint_prep_review_path, sep='\t', index=False)
overlap_audit_df.to_csv(overlap_audit_review_path, sep='\t', index=False)
summary_df.to_csv(summary_review_path, sep='\t', index=False)

patient_header_coverage_df = summary_df.loc[
    summary_df['summary_section'] == 'patient_header_coverage'
].reset_index(drop=True)
followup_version_coverage_df = coverage_df.loc[
    coverage_df['coverage_section'].isin([
        'followup_version_patient_counts',
        'followup_version_record_counts',
        'followup_version_combo_patient_counts',
        'older_version_only_counts',
    ])
].reset_index(drop=True)
gain_and_readiness_df = summary_df.loc[
    summary_df['summary_section'].isin([
        'gain_vs_current_biotab_v4',
        'days_to_death',
        'last_contact_review',
        'os_style_endpoint_prep',
        'status',
    ])
].reset_index(drop=True)

overlap_flag_columns = [
    'has_only_older_xml_followup',
    'has_overlapping_xml_followup_versions',
    'xml_followup_max_last_contact_exceeds_header',
    'xml_header_vs_followup_vital_status_disagreement',
    'xml_header_vs_followup_last_contact_disagreement',
    'xml_v4_vs_current_biotab_v4_last_contact_disagreement',
    'xml_v4_vs_current_biotab_v4_vital_status_disagreement',
    'has_nonmissing_xml_days_to_death',
    'has_os_style_ingredients_any_xml',
    'has_xml_gain_over_biotab_v4',
]
overlap_pattern_df = pd.DataFrame(
    [
        {
            'flag_name': column,
            'patient_count': int((overlap_audit_df[column] == 'true').sum()),
        }
        for column in overlap_flag_columns
    ]
).sort_values(['patient_count', 'flag_name'], ascending=[False, True]).reset_index(drop=True)


In [3]:
print(f"Endpoint-target prep v1 run ID: {latest_pointer['endpoint_target_prep_v1_run_id']}")
print(f"Source run ID: {latest_pointer['source_run_id']}")
print(f"Run log: {run_log_path}")
print(f"Saved: {patient_fields_review_path}")
print(f"Saved: {coverage_review_path}")
print(f"Saved: {endpoint_prep_review_path}")
print(f"Saved: {overlap_audit_review_path}")
print(f"Saved: {summary_review_path}")

display(pd.DataFrame([latest_pointer]))
display(pd.DataFrame([run_log.get('validation', {})]))
display(patient_header_coverage_df)
display(followup_version_coverage_df)
display(gain_and_readiness_df)
display(overlap_pattern_df)
display(endpoint_prep_df.head(10))


Endpoint-target prep v1 run ID: 20260414T025234Z
Source run ID: 20260412T000556Z
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\endpoint-prep\xml_followup_v1_runs\20260414T025234Z\run_log.json
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\97_clinical_xml_patient_endpoint_fields.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\98_clinical_xml_followup_version_coverage.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\99_endpoint_target_prep_v1.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\100_endpoint_target_prep_v1_overlap_audit.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\101_endpoint_target_prep_v1_summary.tsv


,updated_at_utc,endpoint_target_prep_v1_run_id,source_run_id,endpoint_crosswalk_run_id,ambiguity_resolution_run_id,cohort_v1_build_id,baseline_model_input_v1_run_id,baseline_analysis_v1_run_id,processed_run_directory,audit_run_directory,...,endpoint_target_prep_v1_tsv,endpoint_target_prep_v1_overlap_audit_tsv,endpoint_target_prep_v1_summary_tsv,run_log_json,source_supplements_latest_json,endpoint_crosswalk_latest_json,ambiguity_resolution_latest_json,minimal_cohort_v1_latest_json,baseline_model_input_v1_latest_json,baseline_feature_set_v1_audit_map_tsv
0,2026-04-14T02:52:39Z,20260414T025234Z,20260412T000556Z,20260412T042036Z,20260413T085636Z,20260413T202134Z,20260414T013108Z,20260413T213900Z,01-data/processed/tcga-brca/endpoint-prep/xml_...,01-data/audit/tcga-brca/endpoint-prep/xml_foll...,...,01-data/processed/tcga-brca/endpoint-prep/xml_...,01-data/audit/tcga-brca/endpoint-prep/xml_foll...,01-data/audit/tcga-brca/endpoint-prep/xml_foll...,01-data/audit/tcga-brca/endpoint-prep/xml_foll...,01-data/audit/tcga-brca/source/tcga_brca_sourc...,01-data/audit/tcga-brca/variables/tcga_brca_en...,01-data/audit/tcga-brca/cohort/tcga_brca_bluep...,01-data/audit/tcga-brca/cohort/tcga_brca_minim...,01-data/audit/tcga-brca/model-input/tcga_brca_...,01-data/audit/tcga-brca/analysis-prep/baseline...


,passed,required_upstream_pointers_found,source_latest_pointer_found,source_run_log_completed,source_validation_passed,endpoint_crosswalk_latest_pointer_found,endpoint_crosswalk_run_log_completed,endpoint_crosswalk_validation_passed,ambiguity_resolution_latest_pointer_found,ambiguity_resolution_run_log_completed,...,output_rows_positive,endpoint_prep_row_count_matches_minimal_cohort,overlap_audit_row_count_matches_endpoint_prep,row_bridge_fully_matched_to_baseline_feature_set_v1_audit_map,followup_versions_restricted_to_supported_set,unsupported_followup_versions_json,summary_endpoint_freeze_blocked,summary_treatment_unchanged,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,[],True,True,True,True


,endpoint_target_prep_v1_run_id,summary_section,summary_metric,summary_value,notes
0,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_vital_status_count,1097,
1,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_days_to_last_followu...,993,
2,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_days_to_last_known_a...,0,
3,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_days_to_death_count,104,
4,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_person_neoplasm_canc...,972,
5,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_new_tumor_event_afte...,195,
6,20260414T025234Z,patient_header_coverage,patient_header_nonmissing_days_to_new_tumor_ev...,9,


,endpoint_target_prep_v1_run_id,coverage_section,coverage_metric,dimension_name,dimension_value,coverage_value,notes
0,20260414T025234Z,followup_version_patient_counts,patients_with_followup_version,followup_version,1.5,113,
1,20260414T025234Z,followup_version_record_counts,followup_record_count,followup_version,1.5,114,
2,20260414T025234Z,followup_version_patient_counts,patients_with_followup_version,followup_version,2.1,516,
3,20260414T025234Z,followup_version_record_counts,followup_record_count,followup_version,2.1,523,
4,20260414T025234Z,followup_version_patient_counts,patients_with_followup_version,followup_version,4.0,619,
5,20260414T025234Z,followup_version_record_counts,followup_record_count,followup_version,4.0,716,
6,20260414T025234Z,followup_version_combo_patient_counts,patients_with_followup_version_combo,followup_version_combo,1.5,47,
7,20260414T025234Z,followup_version_combo_patient_counts,patients_with_followup_version_combo,followup_version_combo,1.5|2.1,44,
8,20260414T025234Z,followup_version_combo_patient_counts,patients_with_followup_version_combo,followup_version_combo,1.5|2.1|4.0,4,
9,20260414T025234Z,followup_version_combo_patient_counts,patients_with_followup_version_combo,followup_version_combo,1.5|4.0,18,


,endpoint_target_prep_v1_run_id,summary_section,summary_metric,summary_value,notes
0,20260414T025234Z,gain_vs_current_biotab_v4,patients_gaining_followup_beyond_current_biota...,380,Defined as any XML follow-up present with no c...
1,20260414T025234Z,gain_vs_current_biotab_v4,patients_with_xml_gain_over_biotab_v4,443,Union flag covering older-version-only coverag...
2,20260414T025234Z,days_to_death,patients_with_nonmissing_days_to_death_any_xml,151,Counts patient-header or follow-up XML days_to...
3,20260414T025234Z,last_contact_review,patients_with_later_followup_than_patient_head...,746,Compares follow-up max days_to_last_followup a...
4,20260414T025234Z,os_style_endpoint_prep,patients_with_nonmissing_last_contact_like_any...,993,days_to_last_followup is primary comparable fi...
5,20260414T025234Z,os_style_endpoint_prep,patients_with_usable_os_style_ingredients_any_xml,1097,Defined as any XML vital_status plus any XML d...
6,20260414T025234Z,os_style_endpoint_prep,os_style_endpoint_prep_readiness_interpretation,closer_ready_for_manual_endpoint_freeze_review,Endpoint freeze remains blocked; this workflow...
7,20260414T025234Z,status,endpoint_freeze_status,blocked_pending_manual_reconciliation,
8,20260414T025234Z,status,treatment_feasibility_status,unchanged_not_addressed,


,flag_name,patient_count
0,has_os_style_ingredients_any_xml,1097
1,xml_header_vs_followup_last_contact_disagreement,750
2,xml_followup_max_last_contact_exceeds_header,746
3,has_xml_gain_over_biotab_v4,443
4,has_only_older_xml_followup,380
5,has_overlapping_xml_followup_versions,245
6,has_nonmissing_xml_days_to_death,151
7,xml_header_vs_followup_vital_status_disagreement,48
8,xml_v4_vs_current_biotab_v4_last_contact_disag...,0
9,xml_v4_vs_current_biotab_v4_vital_status_disag...,0


,endpoint_target_prep_v1_run_id,cohort_v1_build_id,baseline_model_input_v1_run_id,bcr_patient_barcode,bcr_patient_uuid,provisional_patient_row_id,baseline_analysis_v1_row_id,feature_set_v1_row_index,patient_header_source_label,patient_header_vital_status,...,xml_header_vs_followup_vital_status_disagreement,xml_header_vs_followup_last_contact_disagreement,xml_header_vs_followup_days_to_death_disagreement,xml_followup_max_days_to_last_followup,xml_followup_max_days_to_last_known_alive,xml_followup_max_days_to_death,xml_followup_max_last_contact_exceeds_header,xml_followup_max_last_contact_equals_header,xml_followup_max_last_contact_less_than_header,provisional_endpoint_prep_status_flags_json
0,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-3C-AAAU,6E7D5EC6-A469-467C-B748-237353C23416,1,1,1,patient_header,Alive,...,false,true,false,4047,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
1,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-3C-AALI,55262FCB-1B01-4480-B322-36570430C917,2,2,2,patient_header,Alive,...,false,true,false,4005,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
2,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-3C-AALJ,427D0648-3F77-4FFC-B52C-89855426D647,3,3,3,patient_header,Alive,...,false,true,false,1474,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
3,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-3C-AALK,C31900A4-5DCD-4022-97AC-638E86E889E4,4,4,4,patient_header,Alive,...,false,true,false,1448,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
4,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-4H-AAAK,6623FC5E-00BE-4476-967A-CBD55F676EA6,5,5,5,patient_header,Alive,...,false,true,false,348,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
5,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-5L-AAT0,86C6F993-327F-4525-9983-29C55625593A,6,6,6,patient_header,Alive,...,false,false,false,,,,false,false,false,"[""has_nonmissing_xml_last_contact_like"", ""has_..."
6,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-5L-AAT1,16FC3677-0393-4ED1-AD3F-C8355F056369,7,7,7,patient_header,Alive,...,false,false,false,,,,false,false,false,"[""has_nonmissing_xml_last_contact_like"", ""has_..."
7,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-5T-A9QA,2FD36838-5A83-433E-AC80-B1F77448E5AA,8,8,8,patient_header,Alive,...,false,true,false,303,,,true,false,false,"[""has_any_xml_followup"", ""has_nonmissing_xml_l..."
8,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-A1-A0SB,0045349C-69D9-4306-A403-C9C1FA836644,9,9,9,patient_header,Alive,...,false,false,false,,,,false,false,false,"[""has_nonmissing_xml_last_contact_like"", ""has_..."
9,20260414T025234Z,20260413T202134Z,20260414T013108Z,TCGA-A1-A0SD,C462E422-EB8D-4DAF-9897-2A9C6CBD783A,10,10,10,patient_header,Alive,...,false,false,false,,,,false,false,false,"[""has_nonmissing_xml_last_contact_like"", ""has_..."
